In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', None)

In [2]:
data_path = Path("../../Data/Original_Data")
mort = 'Complications_and_Deaths-Hospital.csv'
readm = 'Unplanned_Hospital_Visits-Hospital.csv'
safety = 'Healthcare_Associated_Infections-Hospital.csv'
cost = 'Medicare_Hospital_Spending_by_Claim.csv'
hospital = 'Hospital_General_Information.csv'

## All data sets are from the Centers for Medicare & Medicaid Services (CMS)
Complications_and_Deaths-Hospital.csv 
    https://data.cms.gov/provider-data/dataset/ynj2-r877
Unplanned_Hospital_Visits-Hospital.csv
    https://data.cms.gov/provider-data/dataset/632h-zaca
Healthcare_Associated_Infections-Hospital.csv
    https://data.cms.gov/provider-data/dataset/77hc-ibv8
Medicare_Hospital_Spending_by_Claim.csv
    https://data.cms.gov/provider-data/dataset/nrth-mfg3
Hospital_General_Information.csv
    https://data.cms.gov/provider-data/dataset/xubh-q36u

In [3]:
# Reading in the Complications and Deaths file. Two of the measures, COMP_HIP_KNEE and PSI_90, will be combined with the infections
# data to obtain a Safety score. The other measures will be used for the Mortality score.


df_mort_long = pd.read_csv(data_path / mort, usecols = ['Facility ID', 'Measure ID', 'Measure Name', 'Score'])
df_mort_long.columns = (df_mort_long.columns.str.replace(' ', '_'))

df_mort_long = df_mort_long[df_mort_long['Measure_ID'].isin(['COMP_HIP_KNEE',
                                                'MORT_30_AMI',
                                                'MORT_30_CABG',
                                                'MORT_30_COPD',
                                                'MORT_30_HF',
                                                'MORT_30_PN',
                                                'MORT_30_STK',
                                                'PSI_04',
                                                'PSI_90'])]
df_mort_long['Score'] = df_mort_long['Score'].replace('Not Available', np.nan)
df_mort_long = df_mort_long.astype({'Score': 'float'})

In [5]:
df_mort = df_mort_long.pivot(index = 'Facility_ID', columns = 'Measure_ID', values = 'Score')
df_mort = df_mort.reset_index()
df_mort = df_mort.set_index('Facility_ID')
df_mort.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4789 entries, 010001 to 671302
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   COMP_HIP_KNEE  1656 non-null   float64
 1   MORT_30_AMI    1956 non-null   float64
 2   MORT_30_CABG   886 non-null    float64
 3   MORT_30_COPD   2660 non-null   float64
 4   MORT_30_HF     3106 non-null   float64
 5   MORT_30_PN     3649 non-null   float64
 6   MORT_30_STK    2217 non-null   float64
 7   PSI_04         1524 non-null   float64
 8   PSI_90         2920 non-null   float64
dtypes: float64(9)
memory usage: 374.1+ KB


In [6]:
# Reading in the Unplanned Hospital Visits file. These measures will be used for the Readmissions score.

df_readm_long = pd.read_csv(data_path / readm, usecols = ['Facility ID', 'Measure ID', 'Measure Name', 'Score'])
df_readm_long.columns = (df_readm_long.columns.str.replace(' ', '_'))
df_readm_long = df_readm_long[df_readm_long['Measure_ID'].isin (['EDAC_30_AMI',
                                                                 'EDAC_30_HF',
                                                                 'EDAC_30_PN',
                                                                 'READM_30_CABG',
                                                                 'READM_30_COPD',
                                                                 'READM_30_HIP_KNEE',
                                                                 'Hybrid_HWR',
                                                                 'OP_32',
                                                                 'OP_35_ADM',
                                                                 'OP_35_ED',
                                                                 'OP_36'])]
df_readm_long['Score'] = df_readm_long['Score'].replace('Not Available', np.nan)
df_readm_long = df_readm_long.astype({'Score': 'float'})

In [7]:
df_readm_long.info()

<class 'pandas.core.frame.DataFrame'>
Index: 52679 entries, 0 to 67044
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Facility_ID   52679 non-null  object 
 1   Measure_ID    52679 non-null  object 
 2   Measure_Name  52679 non-null  object 
 3   Score         26662 non-null  float64
dtypes: float64(1), object(3)
memory usage: 2.0+ MB


In [8]:
df_readm = df_readm_long.pivot(index = 'Facility_ID', columns = 'Measure_ID', values = 'Score')
df_readm = df_readm.reset_index()
df_readm = df_readm.set_index('Facility_ID')

In [9]:
df_readm.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4789 entries, 010001 to 671302
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   EDAC_30_AMI        1555 non-null   float64
 1   EDAC_30_HF         3155 non-null   float64
 2   EDAC_30_PN         3638 non-null   float64
 3   Hybrid_HWR         4217 non-null   float64
 4   OP_32              3152 non-null   float64
 5   OP_35_ADM          1509 non-null   float64
 6   OP_35_ED           1509 non-null   float64
 7   OP_36              2725 non-null   float64
 8   READM_30_CABG      878 non-null    float64
 9   READM_30_COPD      2699 non-null   float64
 10  READM_30_HIP_KNEE  1625 non-null   float64
dtypes: float64(11)
memory usage: 449.0+ KB


In [10]:
# Reading in the Infections file. These measures (along with COMP_HIP_KNEE and PSI_90 from the Complications and Deaths file)
# will be used for the Safety score.

df_safety_long = pd.read_csv(data_path / safety, usecols = ['Facility ID', 'Measure ID', 'Measure Name', 'Score'])
df_safety_long.columns = (df_safety_long.columns.str.replace(' ', '_'))
df_safety_long = df_safety_long[df_safety_long['Measure_ID'].isin(['HAI_1_SIR',
                                                  'HAI_2_SIR',
                                                  'HAI_3_SIR',
                                                  'HAI_4_SIR',
                                                  'HAI_5_SIR',
                                                  'HAI_6_SIR'])]
df_safety_long['Score'] = df_safety_long['Score'].replace('Not Available',np.nan)
df_safety_long = df_safety_long.astype({'Score': 'float'})

In [12]:
df_safety_long.info()

<class 'pandas.core.frame.DataFrame'>
Index: 28734 entries, 5 to 172403
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Facility_ID   28734 non-null  object 
 1   Measure_ID    28734 non-null  object 
 2   Measure_Name  28734 non-null  object 
 3   Score         11272 non-null  float64
dtypes: float64(1), object(3)
memory usage: 1.1+ MB


In [13]:
df_safety = df_safety_long.pivot(index = 'Facility_ID', columns = 'Measure_ID', values = 'Score')
df_safety = df_safety.reset_index()
df_safety = df_safety.set_index('Facility_ID')
#df_safety.head(5)

In [14]:
# Moving COMP_HIP_KNEE and PSI_90 from the mort df to the safety df
safety_cols = ['COMP_HIP_KNEE', 'PSI_90']
df_safety[safety_cols] = df_mort[safety_cols]
df_mort = df_mort.drop(columns = safety_cols) 

In [15]:
df_safety.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4789 entries, 010001 to 671302
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   HAI_1_SIR      1954 non-null   float64
 1   HAI_2_SIR      2237 non-null   float64
 2   HAI_3_SIR      1776 non-null   float64
 3   HAI_4_SIR      616 non-null    float64
 4   HAI_5_SIR      1713 non-null   float64
 5   HAI_6_SIR      2976 non-null   float64
 6   COMP_HIP_KNEE  1656 non-null   float64
 7   PSI_90         2920 non-null   float64
dtypes: float64(8)
memory usage: 336.7+ KB


### Now that the data is separated into the 3 Category Groups, z-scores will be calculated for each measure individually and then a composite category z-score will be calculated for every hospital.  

#### Z-scores are used so that measures with vastly different ranges can be compared with one another
Z-score = (Hospital score - population mean) / population standard deviation.  Because all measures in these groups have smaller values as better, -1 will be multiplied so that all z-scores show higher values as better.

In [16]:
# Z-scores for all individual measures
mort_measures = ['MORT_30_AMI','MORT_30_CABG', 
                    'MORT_30_COPD','MORT_30_HF',
                    'MORT_30_PN','MORT_30_STK','PSI_04']

readm_measures = ['EDAC_30_AMI','EDAC_30_HF',
                     'EDAC_30_PN','READM_30_CABG',
                     'READM_30_COPD','READM_30_HIP_KNEE',
                     'Hybrid_HWR','OP_32','OP_35_ADM',
                     'OP_35_ED','OP_36']

safety_measures = ['HAI_1_SIR','HAI_2_SIR',
                      'HAI_3_SIR','HAI_4_SIR',
                      'HAI_5_SIR','HAI_6_SIR', 
                      'COMP_HIP_KNEE', 'PSI_90']

mort_z = (-1* (df_mort[mort_measures] - df_mort[mort_measures].mean())) / df_mort[mort_measures].std()
mort_z.columns = [m + '_Z' for m in mort_measures]
df_mort = pd.concat([df_mort, mort_z], axis = 1)

readm_z = (-1*(df_readm[readm_measures] - df_readm[readm_measures].mean())) / df_readm[readm_measures].std()
readm_z.columns = [r + '_Z' for r in readm_measures]
df_readm = pd.concat([df_readm, readm_z], axis = 1)

safety_z = (-1*(df_safety[safety_measures] - df_safety[safety_measures].mean())) / df_safety[safety_measures].std()
safety_z.columns = [s + '_Z' for s in safety_measures]
df_safety = pd.concat([df_safety, safety_z], axis = 1)

In [27]:
df_mort.head(5)
#df_readm.head(5)
#df_safety.head(5)

,MORT_30_AMI,MORT_30_CABG,MORT_30_COPD,MORT_30_HF,MORT_30_PN,MORT_30_STK,PSI_04,MORT_30_AMI_Z,MORT_30_CABG_Z,MORT_30_COPD_Z,MORT_30_HF_Z,MORT_30_PN_Z,MORT_30_STK_Z,PSI_04_Z,z_mean_mort,master_mort_z
Facility_ID,,,,,,,,,,,,,,,,
010001,11.4,3.0,9.4,10.2,18.4,13.5,203.00,0.635490,-0.407503,-0.337385,0.630422,-0.740382,-0.155884,-1.255617,-0.232980,-0.222566
010005,NaN,NaN,8.9,14.1,21.2,12.9,184.79,NaN,NaN,-0.014865,-1.156534,-1.753173,0.196643,-0.469844,-0.639554,-0.778592
010006,14.5,5.4,8.7,12.5,19.6,12.4,236.12,-2.026588,-3.278654,0.114143,-0.423424,-1.174436,0.490416,-2.684768,-1.283330,-1.659012
010007,NaN,NaN,11.2,13.4,25.4,NaN,NaN,NaN,NaN,-1.498459,-0.835798,-3.272359,NaN,NaN,-1.868872,-2.459791
010008,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,NaN,NaN,NaN,0.489435,NaN,NaN,0.489435,0.765400


### For each hospital, an average z-score is calculated for each category group

In [18]:

z_mort_measures = ['MORT_30_AMI_Z','MORT_30_CABG_Z', 
                    'MORT_30_COPD_Z','MORT_30_HF_Z',
                    'MORT_30_PN_Z','MORT_30_STK_Z','PSI_04_Z']

z_readm_measures = ['EDAC_30_AMI_Z','EDAC_30_HF_Z',
                     'EDAC_30_PN_Z','READM_30_CABG_Z',
                     'READM_30_COPD_Z','READM_30_HIP_KNEE_Z',
                     'Hybrid_HWR_Z','OP_32_Z','OP_35_ADM_Z',
                     'OP_35_ED_Z','OP_36_Z']

z_safety_measures = ['HAI_1_SIR_Z','HAI_2_SIR_Z',
                      'HAI_3_SIR_Z','HAI_4_SIR_Z',
                      'HAI_5_SIR_Z','HAI_6_SIR_Z', 
                      'COMP_HIP_KNEE_Z', 'PSI_90_Z']

df_mort['z_mean_mort'] = df_mort[z_mort_measures].mean(axis=1)
df_readm['z_mean_readm'] = df_readm[z_readm_measures].mean(axis=1)
df_safety['z_mean_safety'] = df_safety[z_safety_measures].mean(axis=1)


In [19]:
df_safety.head(5)

,HAI_1_SIR,HAI_2_SIR,HAI_3_SIR,HAI_4_SIR,HAI_5_SIR,HAI_6_SIR,COMP_HIP_KNEE,PSI_90,HAI_1_SIR_Z,HAI_2_SIR_Z,HAI_3_SIR_Z,HAI_4_SIR_Z,HAI_5_SIR_Z,HAI_6_SIR_Z,COMP_HIP_KNEE_Z,PSI_90_Z,z_mean_safety
Facility_ID,,,,,,,,,,,,,,,,,
010001,0.418,0.125,0.292,NaN,0.390,0.395,3.2,0.95,0.287385,0.748472,0.802655,NaN,0.512313,0.001343,0.570855,0.242668,0.452242
010005,1.430,1.054,0.804,NaN,2.431,0.430,3.0,0.97,-1.217619,-1.013994,0.071405,NaN,-2.881029,-0.078448,0.845263,0.139862,-0.590651
010006,0.000,0.105,0.000,0.0,0.200,0.102,4.7,1.14,0.909018,0.786415,1.219696,1.16011,0.828205,0.669310,-1.487205,-0.733989,0.418945
010007,NaN,NaN,NaN,NaN,NaN,0.772,NaN,1.06,NaN,NaN,NaN,NaN,NaN,-0.858124,NaN,-0.322765,-0.590444
010008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Now a master z-score is calculated from all of the hospitals' averages

In [20]:
df_mort['master_mort_z'] = (df_mort['z_mean_mort'] - df_mort['z_mean_mort'].mean()) / df_mort['z_mean_mort'].std()
df_readm['master_readm_z'] = (df_readm['z_mean_readm'] - df_readm['z_mean_readm'].mean()) / df_readm['z_mean_readm'].std()
df_safety['master_safety_z'] = (df_safety['z_mean_safety'] - df_safety['z_mean_safety'].mean()) / df_safety['z_mean_safety'].std()

In [42]:
df_mort_new = df_mort['master_mort_z']
df_readm_new = df_readm['master_readm_z']
df_safety_new = df_safety['master_safety_z']

In [29]:
df_hospital = pd.read_csv(data_path / hospital, usecols = ['Facility ID', 'Facility Name',
                                                           'City/Town','State','ZIP Code',
                                                           'Hospital Type','Hospital Ownership',
                                                           'Emergency Services','Count of Facility MORT Measures',
                                                           'Count of Facility Safety Measures',
                                                           'Count of Facility READM Measures',
                                                           'Count of Facility Pt Exp Measures',
                                                           'Count of Facility TE Measures'])
df_hospital.columns = (df_hospital.columns.str.replace(' ', '_').str.replace('/', '_'))

hospital_counters = ['Count_of_Facility_MORT_Measures','Count_of_Facility_Safety_Measures',
                     'Count_of_Facility_READM_Measures','Count_of_Facility_Pt_Exp_Measures',
                     'Count_of_Facility_TE_Measures']

df_hospital[hospital_counters] = df_hospital[hospital_counters].replace('Not Available', np.nan)
df_hospital = df_hospital.astype({'ZIP_Code': 'str', 'Count_of_Facility_MORT_Measures': 'float','Count_of_Facility_Safety_Measures': 'float',
                     'Count_of_Facility_READM_Measures': 'float','Count_of_Facility_Pt_Exp_Measures': 'float',
                     'Count_of_Facility_TE_Measures': 'float'})
df_hospital = df_hospital.set_index('Facility_ID')

df_hospital.head(5)

,Facility_Name,City_Town,State,ZIP_Code,Hospital_Type,Hospital_Ownership,Emergency_Services,Count_of_Facility_MORT_Measures,Count_of_Facility_Safety_Measures,Count_of_Facility_READM_Measures,Count_of_Facility_Pt_Exp_Measures,Count_of_Facility_TE_Measures
Facility_ID,,,,,,,,,,,,
010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,36301,Acute Care Hospitals,Government - Hospital District or Authority,Yes,7.0,7.0,11.0,8.0,11.0
010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,35957,Acute Care Hospitals,Government - Hospital District or Authority,Yes,6.0,7.0,9.0,8.0,12.0
010006,NORTH ALABAMA MEDICAL CENTER,FLORENCE,AL,35630,Acute Care Hospitals,Proprietary,Yes,7.0,8.0,9.0,8.0,10.0
010007,MIZELL MEMORIAL HOSPITAL,OPP,AL,36467,Acute Care Hospitals,Voluntary non-profit - Private,Yes,3.0,3.0,7.0,8.0,7.0
010008,CRENSHAW COMMUNITY HOSPITAL,LUVERNE,AL,36049,Acute Care Hospitals,Proprietary,Yes,1.0,NaN,2.0,NaN,6.0


In [26]:
df_hospital.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5426 entries, 0 to 5425
Data columns (total 13 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Facility_ID                        5426 non-null   object 
 1   Facility_Name                      5426 non-null   object 
 2   City_Town                          5426 non-null   object 
 3   State                              5426 non-null   object 
 4   ZIP_Code                           5426 non-null   object 
 5   Hospital_Type                      5426 non-null   object 
 6   Hospital_Ownership                 5426 non-null   object 
 7   Emergency_Services                 5426 non-null   object 
 8   Count_of_Facility_MORT_Measures    3640 non-null   float64
 9   Count_of_Facility_Safety_Measures  3353 non-null   float64
 10  Count_of_Facility_READM_Measures   4266 non-null   float64
 11  Count_of_Facility_Pt_Exp_Measures  3151 non-null   float

## For the CMS star rating methodology, a hospital is included if it has 3 categories with at least 3 ratings per category, provided either Mortality or Safety (or both) are represented. For this analysis, a hospital is included if it has at least 2 of the 3 categories with at least 3 ratings in the category.

In [36]:
# next step: define if columns meet thresholds for being included
# df['new_col'] = np.where(df['col'] >= 3, 'high', 'low')
df_hospital['Mort'] = np.where(df_hospital['Count_of_Facility_MORT_Measures'] >=3,1,0)
df_hospital['Readm'] = np.where(df_hospital['Count_of_Facility_READM_Measures'] >= 3,1,0)
df_hospital['Safety'] = np.where(df_hospital['Count_of_Facility_Safety_Measures'] >= 3,1,0)
df_hospital['Include'] = df_hospital['Mort'] + df_hospital['Readm'] + df_hospital['Safety']
print(df_hospital['Include'].value_counts())
print(df_hospital.shape)

Include
3    2241
0    1812
1     778
2     595
Name: count, dtype: int64
(5426, 16)


In [40]:
df_hospital_new = df_hospital[df_hospital['Include'] >= 2]
cols_to_drop = ['Count_of_Facility_MORT_Measures', 'Count_of_Facility_Safety_Measures', 'Count_of_Facility_READM_Measures', 
'Count_of_Facility_Pt_Exp_Measures', 'Count_of_Facility_TE_Measures']
df_hospital_new = df_hospital_new.drop(cols_to_drop, axis = 1)
print(df_hospital_new['Include'].value_counts())
print(df_hospital_new.shape)

Include
3    2241
2     595
Name: count, dtype: int64
(2836, 11)


In [46]:
df_hospital_m = pd.merge(df_hospital_new, df_mort_new, left_index = True, right_index = True, how = 'left')
df_hospital_r = pd.merge(df_hospital_m, df_readm_new, left_index = True, right_index = True, how = 'left')
df_hospital_s = pd.merge(df_hospital_r, df_safety_new, left_index = True, right_index = True, how = 'left')
df_hospital_s.head(2)

,Facility_Name,City_Town,State,ZIP_Code,Hospital_Type,Hospital_Ownership,Emergency_Services,Mort,Readm,Safety,Include,master_mort_z,master_readm_z,master_safety_z
Facility_ID,,,,,,,,,,,,,,
010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,36301,Acute Care Hospitals,Government - Hospital District or Authority,Yes,1,1,1,3,-0.222566,0.329290,0.596108
010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,35957,Acute Care Hospitals,Government - Hospital District or Authority,Yes,1,1,1,3,-0.778592,1.544932,-0.707939


In [49]:
# Setting z-scores to NaN where there weren't enough measures in the category

df_hospital_s.loc[df_hospital_s['Mort'] == 0, 'master_mort_z'] = np.nan
df_hospital_s.loc[df_hospital_s['Readm'] == 0, 'master_readm_z'] = np.nan
df_hospital_s.loc[df_hospital_s['Safety'] == 0, 'master_safety_z'] = np.nan
df_hospital_s.head(2)

,Facility_Name,City_Town,State,ZIP_Code,Hospital_Type,Hospital_Ownership,Emergency_Services,Mort,Readm,Safety,Include,master_mort_z,master_readm_z,master_safety_z
Facility_ID,,,,,,,,,,,,,,
010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,36301,Acute Care Hospitals,Government - Hospital District or Authority,Yes,1,1,1,3,-0.222566,0.329290,0.596108
010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,35957,Acute Care Hospitals,Government - Hospital District or Authority,Yes,1,1,1,3,-0.778592,1.544932,-0.707939


In [52]:
df_hospital_s['Outcomes'] = df_hospital_s[['master_mort_z','master_readm_z','master_safety_z']].mean(axis = 1)
df_hospital_s.head(2)

,Facility_Name,City_Town,State,ZIP_Code,Hospital_Type,Hospital_Ownership,Emergency_Services,Mort,Readm,Safety,Include,master_mort_z,master_readm_z,master_safety_z,Outcomes
Facility_ID,,,,,,,,,,,,,,,
010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,36301,Acute Care Hospitals,Government - Hospital District or Authority,Yes,1,1,1,3,-0.222566,0.329290,0.596108,0.234277
010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,35957,Acute Care Hospitals,Government - Hospital District or Authority,Yes,1,1,1,3,-0.778592,1.544932,-0.707939,0.019467
010006,NORTH ALABAMA MEDICAL CENTER,FLORENCE,AL,35630,Acute Care Hospitals,Proprietary,Yes,1,1,1,3,-1.659012,-0.057157,0.554473,-0.387232
010007,MIZELL MEMORIAL HOSPITAL,OPP,AL,36467,Acute Care Hospitals,Voluntary non-profit - Private,Yes,1,1,1,3,-2.459791,-1.215254,-0.707680,-1.460909
010011,ST. VINCENT'S EAST,BIRMINGHAM,AL,35235,Acute Care Hospitals,Voluntary non-profit - Private,Yes,1,1,1,3,-0.411605,-0.208301,-0.003490,-0.207799
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
670122,HOUSTON METHODIST THE WOODLANDS HOSPITAL,THE WOODLANDS,TX,77385,Acute Care Hospitals,Voluntary non-profit - Private,Yes,1,1,1,3,1.451971,0.572714,0.585655,0.870113
670128,BAYLOR SCOTT & WHITE MEDICAL CENTER PFLUGERVILLE,PFLUGERVILLE,TX,78660,Acute Care Hospitals,Voluntary non-profit - Private,Yes,1,1,0,2,0.848584,-0.608480,NaN,0.120052
670132,METHODIST SOUTHLAKE MEDICAL CENTER,SOUTHLAKE,TX,76092,Acute Care Hospitals,Proprietary,Yes,0,1,1,2,NaN,-0.115833,0.681901,0.283034


In [70]:
df_cost = pd.read_csv(data_path / cost)
df_cost.columns = (df_cost.columns.str.replace(' ', '_'))
df_cost['Facility_ID'] = df_cost['Facility_ID'].astype(str).str.zfill(6)
df_cost = df_cost.rename(columns = {'Avg_Spndg_Per_EP_Hospital' : 'Per_Episode_Cost'})
df_cost = df_cost.groupby('Facility_ID')['Per_Episode_Cost'].sum()
#df_cost.shape
df_cost.info()
df_cost

<class 'pandas.core.series.Series'>
Index: 2893 entries, 010001 to 670333
Series name: Per_Episode_Cost
Non-Null Count  Dtype
--------------  -----
2893 non-null   int64
dtypes: int64(1)
memory usage: 45.2+ KB


Facility_ID
010001    53956
010005    47409
010006    51489
010007    40793
010008    42907
          ...  
670265    81384
670266    61345
670300    58819
670309    57838
670333    48796
Name: Per_Episode_Cost, Length: 2893, dtype: int64

In [73]:
df_hospital_cost = pd.merge(df_hospital_s, df_cost, left_index = True, right_index = True, how = 'left')
df_hospital_cost.shape

(2836, 16)

In [74]:
#df_hospital_cost
df_hospital_cost.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2836 entries, 010001 to 670300
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Facility_Name       2836 non-null   object 
 1   City_Town           2836 non-null   object 
 2   State               2836 non-null   object 
 3   ZIP_Code            2836 non-null   object 
 4   Hospital_Type       2836 non-null   object 
 5   Hospital_Ownership  2836 non-null   object 
 6   Emergency_Services  2836 non-null   object 
 7   Mort                2836 non-null   int64  
 8   Readm               2836 non-null   int64  
 9   Safety              2836 non-null   int64  
 10  Include             2836 non-null   int64  
 11  master_mort_z       2725 non-null   float64
 12  master_readm_z      2835 non-null   float64
 13  master_safety_z     2351 non-null   float64
 14  Outcomes            2836 non-null   float64
 15  Per_Episode_Cost    2445 non-null   float64
dtypes: f

## There are 2836 non-null Outcome measures but only 2445 non-null Per_Episode_Cost ones. I need both to come up with a score so I will drop all of the rows that lack values for Per_Episode_Cost.

In [75]:
df_hospital_value = df_hospital_cost[df_hospital_cost['Per_Episode_Cost'].notna()]
df_hospital_value.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2445 entries, 010001 to 670300
Data columns (total 16 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Facility_Name       2445 non-null   object 
 1   City_Town           2445 non-null   object 
 2   State               2445 non-null   object 
 3   ZIP_Code            2445 non-null   object 
 4   Hospital_Type       2445 non-null   object 
 5   Hospital_Ownership  2445 non-null   object 
 6   Emergency_Services  2445 non-null   object 
 7   Mort                2445 non-null   int64  
 8   Readm               2445 non-null   int64  
 9   Safety              2445 non-null   int64  
 10  Include             2445 non-null   int64  
 11  master_mort_z       2338 non-null   float64
 12  master_readm_z      2444 non-null   float64
 13  master_safety_z     2211 non-null   float64
 14  Outcomes            2445 non-null   float64
 15  Per_Episode_Cost    2445 non-null   float64
dtypes: f

In [77]:
cols_to_drop_2 = ['Mort', 'Readm', 'Safety', 'Include', 'master_mort_z', 'master_readm_z', 'master_safety_z']
df_hospital_value = df_hospital_value.drop(cols_to_drop_2, axis = 1)

In [78]:
df_hospital_value.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2445 entries, 010001 to 670300
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Facility_Name       2445 non-null   object 
 1   City_Town           2445 non-null   object 
 2   State               2445 non-null   object 
 3   ZIP_Code            2445 non-null   object 
 4   Hospital_Type       2445 non-null   object 
 5   Hospital_Ownership  2445 non-null   object 
 6   Emergency_Services  2445 non-null   object 
 7   Outcomes            2445 non-null   float64
 8   Per_Episode_Cost    2445 non-null   float64
dtypes: float64(2), object(7)
memory usage: 191.0+ KB


In [79]:
df_hospital_value.head(10)

,Facility_Name,City_Town,State,ZIP_Code,Hospital_Type,Hospital_Ownership,Emergency_Services,Outcomes,Per_Episode_Cost
Facility_ID,,,,,,,,,
010001,SOUTHEAST HEALTH MEDICAL CENTER,DOTHAN,AL,36301,Acute Care Hospitals,Government - Hospital District or Authority,Yes,0.234277,53956.0
010005,MARSHALL MEDICAL CENTERS,BOAZ,AL,35957,Acute Care Hospitals,Government - Hospital District or Authority,Yes,0.019467,47409.0
010006,NORTH ALABAMA MEDICAL CENTER,FLORENCE,AL,35630,Acute Care Hospitals,Proprietary,Yes,-0.387232,51489.0
010007,MIZELL MEMORIAL HOSPITAL,OPP,AL,36467,Acute Care Hospitals,Voluntary non-profit - Private,Yes,-1.460909,40793.0
010011,ST. VINCENT'S EAST,BIRMINGHAM,AL,35235,Acute Care Hospitals,Voluntary non-profit - Private,Yes,-0.207799,47739.0
010012,DEKALB REGIONAL MEDICAL CENTER,FORT PAYNE,AL,35968,Acute Care Hospitals,Proprietary,Yes,-1.091709,47123.0
010016,SHELBY BAPTIST MEDICAL CENTER,ALABASTER,AL,35007,Acute Care Hospitals,Voluntary non-profit - Private,Yes,-0.435222,61043.0
010019,HELEN KELLER HOSPITAL,SHEFFIELD,AL,35660,Acute Care Hospitals,Government - Hospital District or Authority,Yes,-0.734218,47605.0
010021,DALE MEDICAL CENTER,OZARK,AL,36360,Acute Care Hospitals,Government - Hospital District or Authority,Yes,-0.011686,40834.0
